In [4]:
import pandas as pd
import numpy as np
import lfp.lfp_analysis.LFP_collection as LFP_collection
import lfp.lfp_analysis.plotting as lfplt
import lfp.lfp_analysis.event_extraction as ee
import pickle
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import ScalarFormatter
from itertools import combinations, permutations
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from importlib import reload
from collections import defaultdict

def unpickle_this(pickle_file):
    """
    Unpickles things
    Args (1):
        file_name: str, pickle filename that already exists and ends with .pkl
    Returns:
        pickled item
    """
    with open(pickle_file, "rb") as file:
        return pickle.load(file)
    


In [5]:

only_subjects_lfp_json = r"C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Only_Subjects (phase 5)\lfp_data\lfp_collection.json"

only_subjects_collection = LFP_collection.LFPCollection.load_collection(only_subjects_lfp_json)




In [6]:
behavior_dicts = unpickle_this('pilot2/only_subjects/behavior_dict_by_frames.pkl')

In [7]:
exclusion_dict = {'1.1': [],
                  '1.2': [],
                  '1.3': ['BLA'],
                  '2.1': [],
                  '2.2': ['vHPC'],
                  '2.3': ['mPFC', 'vHPC'],
                  '2.4': ['BLA', 'MD'],
                  '3.1': ['vHPC'],
                  '3.2': [],
                  '3.3': ['MD', 'NAc', 'vHPC'],
                  '4.1': [],
                  '4.4': ['NAc']}
only_subjects_collection.exclude_regions(exclusion_dict)

In [8]:
only_subjects_collection.interpolate()

In [9]:
for recording in only_subjects_collection.recordings:
    subject = str(int(recording.name.split('_')[0])/10)
    recording_pattern = (recording.name.split('_')[0] + '_' +
                         recording.name.split('_')[1] + '_' +
                         recording.name.split('_')[2])
    print(recording_pattern)
    recording.event_dict = behavior_dicts[recording_pattern]
    recording.subject = subject
    recording.subject = subject
only_subjects_collection.brain_region_dict


11_CNF_merged.rec
11_NCF_merged.rec
12_CNF_merged.rec
13_NCF_merged.rec
21_FCN_merged.rec
21_NCF_merged.rec
22_FCN_merged.rec
22_NCF_merged.rec
23_CNF_merged.rec
23_NFC_merged.rec
24_CNF_merged.rec
24_NFC_merged.rec
31_NFC_merged.rec
32_NFC_merged.rec
41_FCN_merged.rec
44_FCN_merged.rec


bidict({'mPFC': 0, 'NAc': 1, 'MD': 2, 'BLA': 3, 'vHPC': 4})

In [10]:
def expected_nan_pct(excluded_regions, n_regions=5):
    """
    Calculate expected NaN percentage based on excluded regions.
    
    For power: NaN for each excluded region's channel
    For coherence/granger: NaN for any pair involving an excluded region
    """
    n_excluded = len(excluded_regions)
    
    # Power: one value per region
    power_nan = n_excluded / n_regions * 100
    
    # Coherence/Granger: n×n matrix, diagonal already NaN
    # Count pairs where at least one region is excluded
    total_directed_pairs = n_regions * (n_regions - 1)  # off-diagonal
    # Pairs involving excluded regions (from OR to excluded)
    nan_pairs = n_excluded * (n_regions - 1) * 2 - n_excluded * (n_excluded - 1)
    # Subtract double-counted excluded-to-excluded pairs
    conn_nan = nan_pairs / total_directed_pairs * 100
    
    return power_nan, conn_nan

for recording in only_subjects_collection.recordings:
    n_regions = len(recording.brain_region_dict)
    
    exp_power_nan, exp_conn_nan = expected_nan_pct(
        recording.excluded_regions, 
        n_regions=n_regions
    )
    
    actual_power_nan    = np.mean(np.isnan(recording.power))    * 100
    actual_coherence_nan = np.mean(np.isnan(recording.coherence)) * 100
    actual_granger_nan  = np.mean(np.isnan(recording.granger))  * 100
    
    # Flag subjects where actual NaN exceeds expected
    power_flag    = "⚠️" if actual_power_nan    > exp_power_nan    + 5 else "✓"
    coherence_flag = "⚠️" if actual_coherence_nan > exp_conn_nan     + 5 else "✓"
    granger_flag  = "⚠️" if actual_granger_nan  > exp_conn_nan     + 5 else "✓"
    
    print(
        f"Subject: {recording.subject}  "
        f"Excluded: {recording.excluded_regions}  \n"
        f"  Power:     actual={actual_power_nan:.1f}%  "
        f"expected={exp_power_nan:.1f}%  {power_flag}\n"
        f"  Coherence: actual={actual_coherence_nan:.1f}%  "
        f"expected={exp_conn_nan:.1f}%  {coherence_flag}\n"
        f"  Granger:   actual={actual_granger_nan:.1f}%  "
        f"expected={exp_conn_nan:.1f}%  {granger_flag}\n"
    )

Subject: 1.1  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 1.1  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 1.2  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 1.3  Excluded: ['BLA']  
  Power:     actual=20.0%  expected=20.0%  ✓
  Coherence: actual=32.0%  expected=40.0%  ✓
  Granger:   actual=32.0%  expected=40.0%  ✓

Subject: 2.1  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 2.1  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 2.2  Excluded: ['vHPC']  


In [11]:
import itertools
reload(ee)
regions_bidict = only_subjects_collection.brain_region_dict
averages = {}
events = ['cagemate','novel', 'familiar']
baselines = ['baseline', 'baseline', 'baseline']

def create_nested_dict():
    return defaultdict(lambda: defaultdict(lambda: 
                                           defaultdict(lambda: 
                                                       defaultdict(lambda: 
                                                                   defaultdict((lambda: defaultdict(list)))))))

averages = create_nested_dict()

for recording in only_subjects_collection.recordings: 
    subject = recording.subject
    for mode in ['power', 'coherence', 'granger']:
        regions = list(regions_bidict.keys())
        event_avg_dict = ee.baselined_events(recording, events=events, mode=mode, baseline=baselines)
        agent_band_dict  = ee.band_calcs(event_avg_dict, outer_dict = 'agent', freq_axis = 1)

        for event, bands in agent_band_dict.items():
            for band in bands.keys():
                #print(bands[band][0].shape) #[trials, region]
                band_average = bands[band][0]
                
                if mode == 'power':
                    for brain in regions:
                        reg_idx = only_subjects_collection.brain_region_dict[brain]
                        # Store all trials for this region
                        trials_data = [band_average[trial][reg_idx] for trial in range(band_average.shape[0])]
                        averages[recording.name][subject][event][band]['power'][brain] = trials_data
                
                elif mode == 'coherence':           
                    for region1, region2 in itertools.combinations(regions, 2):
                        idx1 = only_subjects_collection.brain_region_dict[region1]
                        idx2 = only_subjects_collection.brain_region_dict[region2]
                        region_pair = f"{region1}_{region2}"
                        
                        # Store all trials for this region pair
                        trials_data = [band_average[trial, idx1, idx2] for trial in range(band_average.shape[0])]
                        
                        averages[recording.name][subject][event][band]['coherence'][region_pair] = trials_data
                
                elif mode == 'granger':
                    for from_region, to_region in itertools.permutations(regions, 2):
                        from_idx = only_subjects_collection.brain_region_dict[from_region]
                        to_idx = only_subjects_collection.brain_region_dict[to_region]
                        region_pair = f"{from_region}_to_{to_region}"
                        
                        # Store all trials for this directed region pair
                        trials_data = [band_average[trial, to_idx, from_idx] for trial in range(band_average.shape[0])]
                        
                        averages[recording.name][subject][event][band]['granger'][region_pair] = trials_data


def nested_dict_to_long_df(nested_dict):
    """
    Convert nested dictionary to long format DataFrame
    Structure: subject -> event -> band -> metric -> region_or_pair -> [trial_values]
    """
    rows = []
    for recording in nested_dict.keys():
        for subject in nested_dict[recording].keys():
            for event in nested_dict[recording][subject].keys():
                for band in nested_dict[recording][subject][event].keys():
                    for metric in nested_dict[recording][subject][event][band].keys():
                        for region_or_pair in nested_dict[recording][subject][event][band][metric].keys():
                            trial_values = nested_dict[recording][subject][event][band][metric][region_or_pair]
                            # Create a row for each trial
                            for trial_idx, value in enumerate(trial_values):
                                row = {
                                    'subject': subject,
                                    'recording': recording,
                                    'event': event,
                                    'band': band,
                                    'metric': metric,
                                    'region_or_pair': region_or_pair,
                                    'trial': trial_idx,
                                    'value': value
                                }
                                rows.append(row)
        
    return pd.DataFrame(rows)

# Convert to long dataframe
df_long = nested_dict_to_long_df(averages)
df_long
#df_long.to_csv("pilot2/only_subjects/ultra_long_neural_data_only_subjects.csv")

c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:115: RuntimeWarning: Mean of empty slice
  event_snippet = np.nanmean(event_snippet, axis=0)
c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:182: RuntimeWarning: Mean of empty slice
  baseline_recording = np.nanmean(np.array(baseline_averages), axis=0)
c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:332: RuntimeWarning: Mean of empty slice
  band_mean = np.nanmean(arr[tuple(slices)], axis=freq_axis)


,subject,recording,event,band,metric,region_or_pair,trial,value
0,1.1,11_CNF_merged.rec,cagemate,Delta,power,mPFC,0,-36.142363
1,1.1,11_CNF_merged.rec,cagemate,Delta,power,mPFC,1,-11.914498
2,1.1,11_CNF_merged.rec,cagemate,Delta,power,mPFC,2,-74.805193
3,1.1,11_CNF_merged.rec,cagemate,Delta,power,mPFC,3,-28.598526
4,1.1,11_CNF_merged.rec,cagemate,Delta,power,mPFC,4,-62.094465
...,...,...,...,...,...,...,...,...
193020,4.4,44_FCN_merged.rec,familiar,High gamma,granger,vHPC_to_BLA,16,-16.949816
193021,4.4,44_FCN_merged.rec,familiar,High gamma,granger,vHPC_to_BLA,17,23.312168
193022,4.4,44_FCN_merged.rec,familiar,High gamma,granger,vHPC_to_BLA,18,-46.647789
193023,4.4,44_FCN_merged.rec,familiar,High gamma,granger,vHPC_to_BLA,19,0.948387


In [12]:
import numpy as np
import pandas as pd

rows = []

for subject, events_dict in averages.items():
    for event, bands_dict in events_dict.items():
        for band, metrics_dict in bands_dict.items():
            for metric, regions_dict in metrics_dict.items():
                
                # skip gen_baseline metrics
                if 'gen_baseline' in metric:
                    continue
                
                for region_or_pair, trial_values in regions_dict.items():
                    n_trials = len(trial_values)
                    n_nan    = sum(1 for v in trial_values 
                                   if v is None or 
                                   (isinstance(v, float) and np.isnan(v)))
                    
                    rows.append({
                        'subject'        : subject,
                        'metric'         : metric,
                        'region_or_pair' : region_or_pair,
                        'n_trials'       : n_trials,
                        'n_nan'          : n_nan,
                    })

nan_df = pd.DataFrame(rows)

# Collapse across events and bands — sum trials and nans
summary = (nan_df
    .groupby(['subject', 'metric', 'region_or_pair'])
    .agg(
        total_trials = ('n_trials', 'sum'),
        total_nan    = ('n_nan',    'sum')
    )
    .assign(pct_nan = lambda x: (x.total_nan / x.total_trials * 100).round(1))
    .reset_index()
)

print(summary.to_string(index=False))

# Flag anything with any NaN
print("\nPairs with any NaN:")
print(summary[summary.total_nan > 0].to_string(index=False))

          subject     metric region_or_pair  total_trials  total_nan  pct_nan
11_CNF_merged.rec       Beta      coherence            30          0      0.0
11_CNF_merged.rec       Beta        granger            60          0      0.0
11_CNF_merged.rec       Beta          power            15          0      0.0
11_CNF_merged.rec      Delta      coherence            30          0      0.0
11_CNF_merged.rec      Delta        granger            60          0      0.0
11_CNF_merged.rec      Delta          power            15          0      0.0
11_CNF_merged.rec High gamma      coherence            30          0      0.0
11_CNF_merged.rec High gamma        granger            60          0      0.0
11_CNF_merged.rec High gamma          power            15          0      0.0
11_CNF_merged.rec  Low gamma      coherence            30          0      0.0
11_CNF_merged.rec  Low gamma        granger            60          0      0.0
11_CNF_merged.rec  Low gamma          power            15       

In [13]:
df_long['agent'] = df_long['event']

In [14]:
df_long['subject'] = df_long['subject'].astype(float)

In [30]:
sleap_behavior = pd.read_csv('pilot2/only_subjects/sleap_behavior.csv')
sleap_behavior

,Unnamed: 0,subject,recording,agent,partner,trial,exposure,event_length,distance,velocity_mouse1,velocity_mouse2,moving_angle_mouse1,moving_angle_mouse2,front_orientation_mouse1,front_orientation_mouse2,rump_orientation_mouse1,rump_orientation_mouse2,body_angle_mouse1,body_angle_mouse2
0,0,2.3,23_NFC_merged.rec,novel,3.2,0,1,9978,431.701340,0.120357,0.118977,9.044678,-5.864569,0.519222,1.678954,0.585083,2.498046,3.191687,1.657653
1,1,2.3,23_NFC_merged.rec,novel,3.2,1,1,2010,257.196585,0.299902,0.253294,1.374700,-2.935023,1.618142,2.320746,0.687291,1.675650,1.761724,0.016284
2,2,2.3,23_NFC_merged.rec,novel,3.2,2,1,3049,316.406091,0.317216,0.580471,-9.655024,9.183773,1.673550,2.700653,1.227645,2.220513,0.942931,0.568246
3,3,2.3,23_NFC_merged.rec,novel,3.2,3,1,6999,344.563828,0.149628,0.103475,-10.269891,1.617449,1.881502,0.845160,3.677319,0.649974,1.991369,2.377755
4,4,2.3,23_NFC_merged.rec,novel,3.2,4,1,970,268.235734,0.546504,0.162638,-38.120167,1.318554,3.089591,1.307422,2.793402,1.243996,2.482174,3.006899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1099,1099,2.3,23_CNF_merged.rec,novel,1.2,22,2,554,238.250714,0.159840,0.299111,-10.442485,17.267959,3.047660,2.145300,3.469290,1.568868,0.000000,3.119182
1100,1100,2.3,23_CNF_merged.rec,novel,1.2,23,2,554,271.279276,0.495140,0.041056,8.405286,8.773788,1.243815,0.760771,0.596995,0.053744,1.723003,3.136202
1101,1101,2.3,23_CNF_merged.rec,novel,1.2,24,2,4850,292.474962,0.177469,0.121109,-0.402415,-8.866187,0.550421,3.611890,0.864882,3.631243,2.434375,3.437072
1102,1102,2.3,23_CNF_merged.rec,novel,1.2,25,2,2009,408.157935,0.131198,0.051404,-3.972690,-1.844152,0.345251,1.973148,0.818121,4.111999,3.018743,3.916369


In [31]:
df_merged = sleap_behavior.merge(df_long, on=['subject', 'recording', 'agent', 'trial']) 

In [17]:
df_merged

,Unnamed: 0,subject,recording,agent,partner,trial,exposure,distance,velocity_mouse1,velocity_mouse2,...,front_orientation_mouse2,rump_orientation_mouse1,rump_orientation_mouse2,body_angle_mouse1,body_angle_mouse2,event,band,metric,region_or_pair,value
0,0,2.3,23_NFC_merged.rec,novel,3.2,0,1,431.701340,0.120357,0.118977,...,1.678954,0.585083,2.498046,3.191687,1.657653,novel,Delta,power,mPFC,NaN
1,0,2.3,23_NFC_merged.rec,novel,3.2,0,1,431.701340,0.120357,0.118977,...,1.678954,0.585083,2.498046,3.191687,1.657653,novel,Delta,power,NAc,-14.068625
2,0,2.3,23_NFC_merged.rec,novel,3.2,0,1,431.701340,0.120357,0.118977,...,1.678954,0.585083,2.498046,3.191687,1.657653,novel,Delta,power,MD,-2.687805
3,0,2.3,23_NFC_merged.rec,novel,3.2,0,1,431.701340,0.120357,0.118977,...,1.678954,0.585083,2.498046,3.191687,1.657653,novel,Delta,power,BLA,-38.863086
4,0,2.3,23_NFC_merged.rec,novel,3.2,0,1,431.701340,0.120357,0.118977,...,1.678954,0.585083,2.498046,3.191687,1.657653,novel,Delta,power,vHPC,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193020,1103,2.3,23_CNF_merged.rec,novel,1.2,26,2,385.626946,0.154959,0.084092,...,4.080529,1.073034,4.177667,2.812760,3.806335,novel,High gamma,granger,BLA_to_vHPC,NaN
193021,1103,2.3,23_CNF_merged.rec,novel,1.2,26,2,385.626946,0.154959,0.084092,...,4.080529,1.073034,4.177667,2.812760,3.806335,novel,High gamma,granger,vHPC_to_mPFC,NaN
193022,1103,2.3,23_CNF_merged.rec,novel,1.2,26,2,385.626946,0.154959,0.084092,...,4.080529,1.073034,4.177667,2.812760,3.806335,novel,High gamma,granger,vHPC_to_NAc,NaN
193023,1103,2.3,23_CNF_merged.rec,novel,1.2,26,2,385.626946,0.154959,0.084092,...,4.080529,1.073034,4.177667,2.812760,3.806335,novel,High gamma,granger,vHPC_to_MD,NaN


In [22]:
paired_sleap_dicts = unpickle_this('pilot2/only_subjects/paired_sleap_dicts.pkl')

In [29]:
paired_sleap_dicts['11_CNF_merged.rec']['cagemate'].keys()

dict_keys(['locations_1ms', 'time_ms', 'subject', 'partner', 'features_1ms', 'feature_legend', 'node_dict'])

In [ ]:
for recording in only_subjects_collection.recordings:
    print(recording.power.shape[0])
    
    print(paired_sleap_dicts[recording.name]['locations_1ms'].shape)

4803


KeyError: 'locations_1ms'

In [32]:
df_merged.to_csv('pilot2/only_subjects/lfp_kinematics_only_subjects.csv')

In [ ]:
# from importlib import reload
# reload(LFP_collection)
# LFP_collection.LFPCollection.save_to_json(only_subjects_collection, r"C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\only_subjects (phase 5)\lfp_data_updated", 
#                                           notes="Updated with excluded regions, interpolation, and behavior dicts.")